# Data Integration and Aggregation

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
.appName('Data Integration and Aggregation') \
.config('spark.executor.memory','6g') \
.config('spark.executor.cores','4') \
.config('spark.executor.instances','2') \
.config('spark.driver.memory','4g') \
.config('spark.driver.maxResultSize','2g') \
.config('spark.sql.shuffle.partitions','64') \
.config('spark.default.parallelism','64') \
.config('spark.sql.adaptive.enabled','true') \
.config('spark.sql.adaptive.coalescePartition.enabled','true') \
.config('spark.sql.autoBroadcastJoinThreshold',20*1024*1024) \
.config('spark.sql.files.maxPartitionBytes','64MB') \
.config('spark.sql.files.openCostInBytes','2MB') \
.config('spark.memory.fraction',0.8) \
.config('spark.memory.storageFraction',0.2) \
.getOrCreate()

26/09/23 05:54:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
# Now this hardcoded the path so we dont do from the stratch again and again
hdfs_path = '/user/manjurdataengineer/olist/'

In [4]:
customers_df = spark.read.csv(hdfs_path + 'olist_customers_dataset.csv',header = True,inferSchema = True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header = True,inferSchema = True)
order_items_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv',header = True,inferSchema = True)
order_payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv',header = True,inferSchema = True)
order_reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv',header = True,inferSchema = True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv',header = True,inferSchema = True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',header = True,inferSchema = True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header = True,inferSchema = True)
category_name_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header = True,inferSchema = True)

In [5]:
# Cache Frequently used Data for Better Performance

orders_df.cache()
customers_df.cache()
order_items_df.cache()

26/09/23 05:55:11 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.
26/09/23 05:55:11 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.
26/09/23 05:55:11 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [6]:
orders_items_joined_df = orders_df.join(order_items_df,'order_id','inner')

In [7]:
orders_items_products_df = orders_items_joined_df.join(products_df,'product_id','inner')

In [8]:
orders_items_products_sellers_df = orders_items_products_df.join(sellers_df,'seller_id','inner')

In [9]:
full_orders_df = orders_items_products_sellers_df.join(customers_df,'customer_id','inner')

26/09/23 05:55:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [10]:
# GEolocation Data

full_orders_df = full_orders_df.join(geolocation_df,full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,'left')

In [11]:
full_orders_df = full_orders_df.join(order_reviews_df,'order_id','left')

In [12]:
full_orders_df = full_orders_df.join(order_payments_df,'order_id','left')

In [13]:
full_orders_df.cache()

26/09/23 05:55:11 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [14]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [15]:
# Total Revenue Per Seller
from pyspark.sql.functions import *
total_revenue_per_seller = full_orders_df.groupBy('seller_id').agg(sum('price').alias('total_sales')).orderBy('total_sales',ascending=False)


In [16]:
total_revenue_per_seller.show()

+--------------------+--------------------+
|           seller_id|         total_sales|
+--------------------+--------------------+
|4869f7a5dfa277a7d...| 3.613871732000061E7|
|53243585a1d6dc264...| 3.429159295000003E7|
|4a3ca9315b744ce9f...| 3.375957084000195E7|
|7c67e1448b00f6e96...| 3.228232178999851E7|
|fa1c13f2614d7b5c4...|3.0139386310000412E7|
|da8622b14eb17ae28...|2.9857669730001613E7|
|7e93a43ef30c4f03f...|2.6315706299999893E7|
|1025f0e2d44d7041d...|2.2937518519999985E7|
|46dc3b2cc0980fb8e...|2.1791773289999586E7|
|955fee9216a65b617...| 2.096441067000022E7|
|7a67c85e85bb2ce85...|2.0312794889999237E7|
|620c87c171fb2a6dd...|2.0119839600000594E7|
|7d13fca1522535862...|1.8156881909999996E7|
|a1043bafd471dff53...| 1.766267597999973E7|
|6560211a19b47992c...|        1.73159329E7|
|edb1ef5e36e0c8cd8...|1.6624835149999954E7|
|1f50f920176fa81da...| 1.649745443999998E7|
|5dceca129747e92ff...|1.4910548340000035E7|
|cc419e0650a3c5ba7...|1.4751464499999305E7|
|3d871de0142ce09b7...|1.41845252

In [17]:
# Optimized joins for Data Integration

In [18]:
from pyspark.sql.functions import *

In [19]:
orders_items_joined_df = orders_df.join(order_items_df,'order_id','inner')

In [20]:
orders_items_products_df = orders_items_joined_df.join(products_df,'product_id','inner')

In [21]:
# Basically here we use the broadcast join on that dataframe which are small so they give that to every executor so on thier
# On they can run so becauase of that suffle time reduce
orders_items_products_sellers_df = orders_items_products_df.join(broadcast(sellers_df),'seller_id','inner')

In [22]:
full_orders_df = orders_items_products_sellers_df.join(customers_df,'customer_id','inner')

In [23]:
# GEolocation Data

full_orders_df = full_orders_df.join(broadcast(geolocation_df),full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,'left')

In [24]:
full_orders_df = full_orders_df.join(broadcast(order_reviews_df),'order_id','left')

In [25]:
full_orders_df = full_orders_df.join(order_payments_df,'order_id','left')

In [26]:
full_orders_df.cache()

26/09/23 05:56:18 WARN CacheManager: Asked to cache already cached data.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: int, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: string, review_comment_title: string, review_commen

In [27]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [28]:
# Total Orders Per Customer

In [29]:
customer_order_count_df = full_orders_df.groupBy('customer_id')\
.agg(count('order_id').alias('total_orders'))\
.orderBy(desc('total_orders'))

In [30]:
customer_order_count_df.show()

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|351e40989da90e704...|       11427|
|50920f8cd0681fd86...|       10752|
|9b43e2a62de9bab3a...|        8556|
|270c23a11d024a44c...|        8001|
|d3e82ccec3cb5f956...|        6876|
|5c87184371002d49e...|        6876|
|d5f2b3f597c7ccafb...|        6706|
|c2f18647725395af4...|        6612|
|24e7dc2ff8c071263...|        6597|
|7bb57d182bdc11653...|        6258|
|d22f25a9fadfb1abb...|        6072|
|63b964e79dee32a35...|        6072|
|1ff773612ab8934db...|        5820|
|13aa59158da63ba0e...|        5206|
|78fc46047c4a639e8...|        5200|
|dd3f1762eb601f41c...|        4992|
|a193aa8d905b8e246...|        4896|
|9eb3d566e87289dcb...|        4872|
|2ba91e12e5e4c9f56...|        4752|
|1b2ab6eda1946a6ff...|        4728|
+--------------------+------------+
only showing top 20 rows



In [31]:
# Average review score per seller

In [32]:
avg_review_score_per_seller_df = full_orders_df.groupBy('seller_id')\
.agg(avg('review_score').alias('Avg_Review_Score'))\
.orderBy(desc('Avg_Review_Score'))

In [33]:
avg_review_score_per_seller_df.show()

+--------------------+----------------+
|           seller_id|Avg_Review_Score|
+--------------------+----------------+
|52b53f7061969fe47...|             5.0|
|64c9a1db4e73e19aa...|             5.0|
|5587130f9850696ed...|             5.0|
|964ff64dc05b13811...|             5.0|
|2a73cba571d90c694...|             5.0|
|ba89796db68db180b...|             5.0|
|5305693ffae2d3463...|             5.0|
|293dee954235e4e4e...|             5.0|
|544863df86ed3b78a...|             5.0|
|9c1c0c36cd23c2089...|             5.0|
|3bfba5a77604f6dfa...|             5.0|
|9b1f656ca3060f8af...|             5.0|
|d2572f31e9023e985...|             5.0|
|1c742ac33582852aa...|             5.0|
|e560e4022d8f7aebf...|             5.0|
|186cdd1b2df32caa7...|             5.0|
|83c8aafbefb034674...|             5.0|
|0570350b23eda5444...|             5.0|
|ec933281fb017b502...|             5.0|
|5db42a6cd9bf1bf0d...|             5.0|
+--------------------+----------------+
only showing top 20 rows



In [34]:
# Top 10 Most Sold Products

top_10_products = full_orders_df.groupBy('product_id')\
.agg(count('order_id').alias('total_sold'),sum('price').alias('total_sales'))\
.orderBy(desc('total_sold'))\
.limit(10)




In [35]:
top_10_products.show()

+--------------------+----------+------------------+
|          product_id|total_sold|       total_sales|
+--------------------+----------+------------------+
|aca2eb7d00ea1a7b8...|     86740| 6164630.299999952|
|422879e10f4668299...|     81110| 4442791.509999977|
|99a4788cb24856965...|     78775|6921762.7099999115|
|389d119b48cf3043d...|     60248|3280533.1299999696|
|d1c427060a0f73f6b...|     59274| 8220103.329999958|
|368c6c730842d7801...|     58358|3181698.8999999627|
|53759a2ecddad2bb8...|     52654|2893017.4999999804|
|53b36df67ebb7c415...|     52105| 6159887.410000005|
|154e7e31ebfa09220...|     42700| 962161.0000000079|
|3dd2a17168ec895c7...|     40787| 6116941.300000025|
+--------------------+----------+------------------+



# Window Function and Ranking

In [36]:
from pyspark.sql.window import Window


In [37]:
# Per Customer Spending
customer_spending = full_orders_df.groupBy('customer_id').agg(sum('price').alias('Total_Spending')).orderBy(desc('Total_Spending'))

In [38]:
customer_spending.show()

+--------------------+------------------+
|         customer_id|    Total_Spending|
+--------------------+------------------+
|d3e82ccec3cb5f956...|         6662844.0|
|df55c14d1476a9a34...|         3565657.0|
|fe5113a38e3575c04...|         3293604.0|
|ec5b2ba62e5743423...|         2556120.0|
|63b964e79dee32a35...|         2501664.0|
|46bb3c0b1a65c8399...|         2336752.0|
|05455dfa7cd02f13d...| 2160194.400000087|
|3690e975641f01bd0...|         2124498.0|
|349509b216bd5ec11...|         1923627.0|
|695476b5848d64ba0...|1820543.1299999943|
|73236a0796f53d60d...|         1755520.0|
|cc803a2c412833101...|         1676400.0|
|1ff773612ab8934db...|1658641.7999999512|
|fced842c7dad61e8c...|         1654898.0|
|1ecb47d23dc8203cd...|1629588.3599999903|
|de832e8dbb1f588a4...|1584990.5999999817|
|803cd9b04f9cd252c...|         1512312.0|
|d72181923840c8895...|1488114.8999999566|
|06d478ba352a27a51...|         1461150.0|
|0049e8442c2a3e4a8...|         1444800.0|
+--------------------+------------

In [39]:
# Dense Rank for Sellers Based on Revenue
window_spec = Window.partitionBy('seller_id').orderBy(desc('price'))

In [40]:
# rank top 3 selling products per seller
from pyspark.sql.window import Window
frfull_orders_dfyspark.sql.functions import count, desc, row_number

# 1. Calculate total units sold for each seller-product combination
seller_product_sales = full_orders_df.groupBy(
    'seller_id', 'product_id'
).agg(
    count('order_id').alias('total_sold')
)

# 2. Create a window for each seller
window_spec = Window.partitionBy('seller_id') \
    .orderBy(desc('total_sold'))

# 3. Rank products within each seller
ranked_products = seller_product_sales.withColumn(
    'rank',
    row_number().over(window_spec)
)

# 4. Keep only top 3 products per seller
top_3_products_per_seller = ranked_products.filter(
    col('rank') <= 3
)

top_3_products_per_seller.show()

SyntaxError: invalid syntax (2400094839.py, line 3)

In [41]:
# Total Revenue & Average Order Value (AOV) per Customer

total_revenue_aov_df = full_orders_df.groupBy('customer_id')\
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_spend'),
    round(avg('price'),2).alias('AOV')
)\
.orderBy(desc('total_spend'))


In [42]:
total_revenue_aov_df.show()

+--------------------+------------+------------------+-------+
|         customer_id|total_orders|       total_spend|    AOV|
+--------------------+------------+------------------+-------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|
|695476b5848d64ba0...|         687|1820543.1299999943|2649.99|
|73236a0796f53d60d...|         832|         1755520.0| 2110.0|
|cc803a2c412833101...|         762|         1676400.0| 2200.0|
|1ff773612ab8934db...|        5820|1658641.7999999512| 

In [43]:
# Seller Performance Metrics ( Revenue, Average Review, Order Count)
seller_performance_metric_df = full_orders_df.groupBy('seller_id')\
.agg(
    sum('price').alias('Revenue'),
    round(avg('review_score'),2).alias('avg_review_score'),
    count('order_id').alias('Order Count'),
    round(stddev('price'),2).alias('price_variability')
)\
.orderBy(desc('Revenue'))

In [44]:
seller_performance_metric_df.show()

+--------------------+--------------------+----------------+-----------+-----------------+
|           seller_id|             Revenue|avg_review_score|Order Count|price_variability|
+--------------------+--------------------+----------------+-----------+-----------------+
|4869f7a5dfa277a7d...|3.6138717320000626E7|            4.09|     184587|           111.65|
|53243585a1d6dc264...|3.4291592950000025E7|            4.12|      54514|           499.65|
|4a3ca9315b744ce9f...| 3.375957084000194E7|            3.77|     330661|            59.37|
|7c67e1448b00f6e96...|3.2282321789998513E7|            3.42|     233306|            50.39|
|fa1c13f2614d7b5c4...|3.0139386310000412E7|            4.38|      87686|            307.7|
|da8622b14eb17ae28...|  2.98576697300016E7|            3.98|     264433|            72.92|
|7e93a43ef30c4f03f...|2.6315706299999885E7|            4.15|      50226|           377.24|
|1025f0e2d44d7041d...|2.2937518519999992E7|            3.89|     229587|             84.3|

In [45]:
# Product Popularity Metrics
products_metric = full_orders_df.groupBy('product_id')\
.agg(
    count('order_id').alias('Total_Sales'),
    sum('price').alias('Total_Revenue'),
    round(avg('price'),2).alias('avg_price'),
    round(stddev('price'),2).alias('price_volatility'),\
    collect_set('seller_id').alias('unique_seller')
)\
.orderBy(desc('Total_Revenue'))

In [46]:
products_metric.show()

+--------------------+-----------+-----------------+---------+----------------+--------------------+
|          product_id|Total_Sales|    Total_Revenue|avg_price|price_volatility|       unique_seller|
+--------------------+-----------+-----------------+---------+----------------+--------------------+
|588531f8ec37e7d5f...|       5732|8301386.529999997|  1448.25|           47.46|[53243585a1d6dc26...|
|d1c427060a0f73f6b...|      59274|8220103.329999957|   138.68|           16.58|[a1043bafd471dff5...|
|bb50f2e236e5eea01...|      23609|        7728975.0|   327.37|             2.5|[f7ba60f8c3f99e7e...|
|cb26d15d1b6eabaac...|       7687|        7413149.0|   964.37|           17.23|[ccb83a794700270f...|
|99a4788cb24856965...|      78775| 6921762.70999992|    87.87|            4.08|[4a3ca9315b744ce9...|
|6cdd53843498f9289...|      19553|6836471.499999987|   349.64|           12.05|[ccc4bbb5f32a6ab2...|
|aca2eb7d00ea1a7b8...|      86740|6164630.299999952|    71.07|            3.17|[955fee9216a

In [47]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [48]:
# Customer Retention Analysis (First & Last Order)
customer_retention_df = full_orders_df.groupBy('customer_unique_id') \
    .agg(
        min('order_purchase_timestamp').alias('first_order_date'),
        max('order_purchase_timestamp').alias('last_order_date'),
        countDistinct('order_id').alias('total_orders')
    ) \
    .withColumn('days_between', datediff('last_order_date', 'first_order_date')) \
    .orderBy(desc('total_orders'))

customer_retention_df.show()

+--------------------+-------------------+-------------------+------------+------------+
|  customer_unique_id|   first_order_date|    last_order_date|total_orders|days_between|
+--------------------+-------------------+-------------------+------------+------------+
|8d50f5eadf50201cc...|2017-05-15 23:30:03|2018-08-20 19:14:26|          16|         462|
|3e43e6105506432c9...|2017-09-18 18:53:15|2018-02-27 18:36:39|           9|         162|
|6469f99c1f9dfae77...|2017-09-19 01:02:44|2018-06-28 00:43:34|           7|         282|
|1b6c7548a2a1f9037...|2017-11-13 16:44:41|2018-02-14 13:22:12|           7|          93|
|ca77025e7201e3b30...|2017-10-09 12:34:39|2018-06-01 11:38:29|           7|         235|
|63cfc61cee11cbe30...|2017-05-11 14:39:53|2018-05-28 17:20:02|           6|         382|
|f0e310a6839dce9de...|2017-05-20 08:53:30|2018-04-05 09:04:45|           6|         320|
|12f5d6e1cbf93dafd...|2017-01-05 14:18:03|2017-01-05 15:25:10|           6|           0|
|47c1a3033b8b77b3a...

In [49]:
# Customer Segmentation based on spending
customer_spending_df = full_orders_df.groupBy('customer_id')\
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_spent'),
    round(avg('price'),2).alias('AOV')
)\
.orderBy(desc('total_spent'))

In [50]:
customer_segmentation_based_on_spending = customer_spending_df.withColumn(
    'customer_segment',
    when(col('AOV') >=1200,"High-Value")
    .when( (col('AOV')<1200) & (col('AOV') >=700),'Medium_Value')
    .otherwise('Low-Value'))

In [51]:
customer_segmentation_based_on_spending.show()

+--------------------+------------+------------------+-------+----------------+
|         customer_id|total_orders|       total_spent|    AOV|customer_segment|
+--------------------+------------+------------------+-------+----------------+
|d3e82ccec3cb5f956...|        6876|         6662844.0|  969.0|    Medium_Value|
|df55c14d1476a9a34...|         743|         3565657.0| 4799.0|      High-Value|
|fe5113a38e3575c04...|        2292|         3293604.0| 1437.0|      High-Value|
|ec5b2ba62e5743423...|        1428|         2556120.0| 1790.0|      High-Value|
|63b964e79dee32a35...|        6072|         2501664.0|  412.0|       Low-Value|
|46bb3c0b1a65c8399...|         748|         2336752.0| 3124.0|      High-Value|
|05455dfa7cd02f13d...|        2184| 2160194.400000087|  989.1|    Medium_Value|
|3690e975641f01bd0...|         802|         2124498.0| 2649.0|      High-Value|
|349509b216bd5ec11...|         743|         1923627.0| 2589.0|      High-Value|
|695476b5848d64ba0...|         687|18205

In [52]:
# Order Volume by Customer State
state_orders_df = full_orders_df.groupBy('customer_state') \
    .agg(
        countDistinct('order_id').alias('total_orders'),
        countDistinct('customer_unique_id').alias('unique_customers')
    ) \
    .orderBy(desc('total_orders'))

state_orders_df.show(30)

+--------------+------------+----------------+
|customer_state|total_orders|unique_customers|
+--------------+------------+----------------+
|            SP|       41375|           39981|
|            RJ|       12762|           12303|
|            MG|       11544|           11178|
|            RS|        5432|            5249|
|            PR|        4998|            4840|
|            SC|        3612|            3513|
|            BA|        3358|            3257|
|            DF|        2125|            2062|
|            ES|        2025|            1956|
|            GO|        2007|            1942|
|            PE|        1648|            1605|
|            CE|        1327|            1305|
|            PA|         970|             944|
|            MT|         903|             873|
|            MA|         740|             719|
|            MS|         709|             688|
|            PB|         532|             516|
|            PI|         493|             481|
|            

In [53]:
# Weekday vs Weekend Order
daily_df = full_orders_df.withColumn('day_name', date_format('order_purchase_timestamp', 'EEEE')) \
    .groupBy('day_name') \
    .agg(countDistinct('order_id').alias('total_orders')) \
    .orderBy(desc('total_orders'))

daily_df.show()

+---------+------------+
| day_name|total_orders|
+---------+------------+
|   Monday|       16068|
|  Tuesday|       15831|
|Wednesday|       15425|
| Thursday|       14639|
|   Friday|       14002|
|   Sunday|       11888|
| Saturday|       10813|
+---------+------------+



In [54]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [62]:
!hadoop fs -mkdir /user/manjurdataengineer/olist/processced_data/

In [65]:
full_orders_df = full_orders_df.withColumn('hour_of_day', hour('order_purchase_timestamp'))

In [66]:
# Hourly Order Distribution
slot_df = full_orders_df.withColumn('time_slot',
        when(col('hour_of_day').between(0, 5), 'Late Night')
        .when(col('hour_of_day').between(6, 11), 'Morning')
        .when(col('hour_of_day').between(12, 17), 'Afternoon')
        .otherwise('Evening')
    ) \
    .groupBy('time_slot') \
    .agg(countDistinct('order_id').alias('total_orders')) \
    .orderBy(desc('total_orders'))

slot_df.show()

26/09/23 06:01:51 INFO PlanChangeLogger: 
 Dataproc Rule org.apache.spark.sql.catalyst.optimizer.google.EncodeStringCaseWhenInAggregate effective 1 times.



+----------+------------+
| time_slot|total_orders|
+----------+------------+
| Afternoon|       38059|
|   Evening|       33864|
|   Morning|       22043|
|Late Night|        4700|
+----------+------------+



In [63]:
# save as Parquet in hdfs

full_orders_df.write.mode('overwrite').parquet('/user/manjurmahamad/olist/processced_data/')